<a href="https://colab.research.google.com/github/LLM-AT-SCALE/claude-agent-sdk-Labs/blob/main/Lab-1/Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Loan Application Evaluation
### Claude Agent SDK + Messages API

In this lab you will score a commercial loan application narrative against the
**Five C's of Credit**, twice, using two different Claude surfaces:

| | Surface | What it does |
|---|---|---|
| **Part A** | Messages API | One call. You specify the task completely; Claude returns structured JSON. |
| **Part B** | Claude Agent SDK | An agent with a folder and tools. It decides its own steps and writes the memo itself. |

Both parts score the same application. The point of the lab is the **difference in shape** —
when a single call is enough, and when you actually need an agent.

---

**Your Anthropic API key is requested at run time.** It is never written to this
notebook, never committed to the repository, and never stored on disk.

## Step 1 — Clone the repository and install dependencies

Run the cell below. It clones only this lab, installs the Python packages, and
installs the **Claude Code CLI** — the Agent SDK in Part B drives that CLI, so
without it Part B raises `CLINotFoundError`.

If you see a **Restart session** dialog while it installs, click **Restart
session**, then run this cell again.

In [ ]:
# Start clean. Without this, a second run of this cell hits
# "destination path already exists", git clone fails, and every cell below
# silently keeps running against the OLD copy of the lab.
%cd /content
!rm -rf /content/claude-agent-sdk-Labs

# Clone only what this lab needs
# --depth 1: only the latest commit          --filter=blob:none: skip file contents
# --sparse: check out one folder rather than the whole repository
!git clone --depth 1 --filter=blob:none --sparse https://github.com/LLM-AT-SCALE/claude-agent-sdk-Labs.git
%cd /content/claude-agent-sdk-Labs
!git sparse-checkout set Lab-1
%cd /content/claude-agent-sdk-Labs/Lab-1

# Python dependencies for the lab
!pip install -q -r requirements.txt

# The Agent SDK (Part B) runs the Claude Code CLI as a subprocess
!npm install -g @anthropic-ai/claude-code

# ngrok exposes the Streamlit app so you can open it from Colab
!pip install -q pyngrok

print("\nFetched commit:")
!git log --oneline -1
print("\nUploader accepts:")
!grep -A2 "Upload a loan application" src/main.py | head -3
print("\nClaude Code CLI:")
!claude --version

## Step 2 — Enter your Anthropic API key

`getpass` hides what you type, so the key never appears in the notebook output
and is never saved with the file.

Get a key from [console.anthropic.com](https://console.anthropic.com/settings/keys).

In [ ]:
import os, getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
print("Key stored for this session only.")

## Step 3 — Check the key works

One cheap call, before the lab spends real tokens. You should see
`Valid API Key!`

In [ ]:
import sys
sys.path.insert(0, ".")

from src.validate import validate_anthropic_key

print(validate_anthropic_key(os.environ["ANTHROPIC_API_KEY"]))

## Step 4 — Part A: score the application with one Messages API call

The task here is fully specified — read this narrative, apply this rubric,
return these fields. That is exactly what a single call is good at.

### Choose the application

Run the cell below and click **Choose Files** to upload your own loan application (PDF, DOCX or TXT). Press **Cancel** to use the built-in sample instead.

Whatever you choose here is what both Part A and Part B score.

In [ ]:
from pathlib import Path
from google.colab import files

# Click 'Choose Files' to upload your own application, or Cancel for the sample.
uploaded = files.upload()

if uploaded:
    name = next(iter(uploaded))
    application = Path("uploads") / name
    application.parent.mkdir(exist_ok=True)
    application.write_bytes(uploaded[name])
else:
    application = Path("data/strong-approve.pdf")

print("Using:", application)

In [ ]:
import json
from pathlib import Path

from src.ingestion import extract_text
from src.model import score_application
from src.scoring import load_rubric, aggregate

rubric = load_rubric()

# `application` was chosen in the cell above - your upload, or the sample.
narrative = extract_text(application.name, application.read_bytes())
print(f"Read {len(narrative)} characters out of {application.name}\n")

result = score_application(narrative, rubric, os.environ["ANTHROPIC_API_KEY"])
print(json.dumps(result, indent=2)[:1500])

### The arithmetic

Claude chose the 1–5 score for each C. The weighting, the total and the decision
band are computed in plain Python, so the maths is identical every run.

In [ ]:
summary = aggregate(result["scores"], rubric)

for row in summary["rows"]:
    score = "N/E" if row["score"] is None else row["score"]
    points = "-" if row["points"] is None else round(row["points"], 1)
    print(f"{row['id']}  {row['name']:<12} score={score:<4} weight={row['weight']:<4} points={points}")

print()
print(f"{summary['earned']} / {summary['evidenced_weight']} x 100 = {summary['raw_total']} -> {summary['total']}")
print(f"DECISION: {summary['decision']}")
if summary["overridden_by"]:
    print(f"(set by hard rule {summary['overridden_by']})")

### Try the other samples

Four applications ship with the lab, **each in `.pdf`, `.docx` and `.txt`**, and
between them they hit every decision band:

| Application | What it tests |
|---|---|
| `strong-approve` | Every C evidenced, comfortable metrics |
| `hard-rule-decline` | DSCR below 1.00x — the hard rule should override the band |
| `incomplete-evidence` | No collateral evidence at all — one C should score N/E |
| `stated-vs-computed-conflict` | Claims a DSCR its own figures contradict |

Re-run the **Choose the application** cell, pick a file from the `data/` folder (left sidebar → folder icon → download it first, or upload your own), then re-run Part A. Watch what the hard rule does.

The same application in `.pdf` and in `.docx` should score identically — the
format is only packaging, and `src/ingestion.py` is what unwraps it.

**There is no OCR.** A scanned or photographed PDF has no text layer, so the lab
rejects it with a message rather than guessing at pixels.

## Step 5 — Part B: the same job, given to an agent

Now hand the application to the **Claude Agent SDK**. Instead of one specified
call, the agent gets a folder and its built-in tools, and decides its own steps:
read the rubric, read the narrative, work through the five C's, write a memo.

Watch the `-> Using tool:` lines — that is the agent loop, made visible.

In [ ]:
import shutil
from pathlib import Path

from src.agent import run_agent

# Give the agent a clean folder holding only what it needs
workspace = Path("outputs")
workspace.mkdir(exist_ok=True)
shutil.copy("rubric.json", workspace / "rubric.json")

# The agent's tools read text, so unwrap the PDF first and hand it the narrative
narrative_path = workspace / "loan-application.txt"
narrative_path.write_text(narrative, encoding="utf-8")

async def main():
    async for kind, text in run_agent(narrative_path, workspace):
        prefix = {"tool": "-> ", "result": "[done] ", "text": ""}[kind]
        print(prefix + text[:400])

# Colab supports top-level await inside a cell
await main()

### Read what the agent wrote

In [ ]:
from IPython.display import Markdown, display

memo = Path("outputs/credit_memo.md")
display(Markdown(memo.read_text(encoding="utf-8")) if memo.exists()
        else Markdown("_The agent did not write a memo. Re-run the cell above._"))

## Step 6 — Run the web application

Both parts, behind a Streamlit interface. The app prompts for the API key in a
password field — the same run-time pattern, in a UI.

Run the cell, then open the printed ngrok URL. **Leave this cell running** while
you use the app; stopping it closes the tunnel.

The cell is safe to re-run — it stops any server left over from a previous run
before starting a new one. If you skip that, the port stays taken, the new
server exits, and the tunnel keeps serving the **old** app, which looks like
the page hanging on *"Please wait…"* with no explanation.

To clear the port by hand at any time:

```
!pkill -f "streamlit run"
```

In [ ]:
import os, time, getpass, subprocess
from pyngrok import ngrok

PORT = 8501

# Stop anything an earlier run of this cell left behind.
#
# This matters more than it looks. Without it, a second run finds the port
# taken, the NEW server exits immediately, and the tunnel quietly keeps
# pointing at the OLD one — which shows up in the browser as "Please wait..."
# forever, with no obvious cause.
subprocess.run("pkill -f 'streamlit run'", shell=True)
ngrok.kill()
time.sleep(2)

# Streamlit asks for an email on first run, which stalls a headless start.
# Writing an empty credentials file skips the prompt.
os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)
with open(os.path.expanduser("~/.streamlit/credentials.toml"), "w") as f:
    f.write('[general]\nemail = ""\n')

# ngrok needs its own free authtoken - https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(getpass.getpass("Enter your ngrok authtoken: "))

# Open the tunnel first, then start the server behind it.
# A free ngrok account allows ONE endpoint. If another Colab session of yours
# is still running this cell, ngrok refuses (ERR_NGROK_334) - say so plainly.
try:
    public_url = ngrok.connect(PORT).public_url
except Exception as exc:
    if "ERR_NGROK_334" in str(exc) or "already online" in str(exc):
        raise SystemExit(
            "
!! Your ngrok endpoint is already in use by an EARLIER session.
"
            "   Go to that other Colab tab and stop its last cell (or Runtime -> Disconnect
"
            "   and delete runtime), then run this cell again.
"
            "   Or stop it at dashboard.ngrok.com -> Agents.")
    raise

# Launch Streamlit, keeping its output somewhere we can read it if it fails
get_ipython().system_raw(f"streamlit run app.py --server.port {PORT} > streamlit.log 2>&1 &")
time.sleep(12)

log = open("streamlit.log").read()
print("--- streamlit.log ---")
print(log[-1200:])

if "already in use" in log:
    print("\n!! The port was still busy, so the app did not start.")
    print("   Run this cell again - it clears the port first.")
else:
    print("\nOpen your app here:", public_url)
    print("Leave this cell running while you use the app; stopping it closes the tunnel.")
    time.sleep(60 * 20)

---

## What to take away

**Use a single Messages API call when you can specify the task.** Extraction,
classification, scoring against a fixed rubric — one call is cheaper, faster and
easier to test.

**Reach for the Agent SDK when the steps aren't knowable in advance.** The agent
reads files, decides what to look at next, and produces an artefact. You pay for
that in latency and tokens, so it has to be earning its place.

**Never hardcode an API key.** Both parts of this lab took the key at run time —
`getpass` in the notebook, a password field in the app. Nothing was written to
disk, and nothing went into the repository.